# XGBoost GBM 最优模型 (B1 MinTrees)

## 模型配置
- **预处理**: P1_raw (不预处理，树模型对单调变换不敏感)
- **目标函数**: IC-aware 自定义目标 (直接优化截面 Pearson 相关)
- **训练策略**: 强制热身 50 棵树 + early stopping
- **正则化**: Phase 2 精调参数 (100 次随机搜索)

## 数据 & 窗口
- 数据: `factortb.parquet`, 230 个 ALPHA 因子, 目标 FWD_RET_10D
- 滚动窗口: 2年训练 / 6个月测试 / 6个月步长 / 15个交易日 gap
- 共 17 个窗口 (2018H1 ~ 2026H1)

## 防泄露设计 (最保守)
- 训练集内部按时间 80/20 分割: 子训练集 + 验证集
- 验证集仅用于 early stopping 选树数量，不参与梯度计算
- **不做全量重训**: 直接使用 early stopping 产出的模型，验证集数据从不流入模型权重
- Gap = 15 个交易日 > FWD_RET_10D (10天)，确保标签无前视泄露

In [2]:
import xgboost as xgb
import numpy as np
import pandas as pd
import time, gc, warnings
warnings.filterwarnings('ignore')

N_JOBS = 24
RANDOM_STATE = 42

print(f'XGBoost: {xgb.__version__}')

XGBoost: 2.1.4


## 1. 数据加载

In [3]:
%%time
df = pd.read_parquet('factortb.parquet')
df = df.sort_values(['trade_date', 'ts_code']).reset_index(drop=True)
feature_cols = [c for c in df.columns if c.startswith('ALPHA')]
target_col = 'FWD_RET_10D'
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)
print(f'Data: {df.shape}, Features: {len(feature_cols)}')
print(f'Date range: {df["trade_date"].min().date()} ~ {df["trade_date"].max().date()}')

Data: (883816, 233), Features: 230
Date range: 2014-01-02 ~ 2026-05-22
CPU times: total: 4.92 s
Wall time: 1.95 s


## 2. 滚动窗口生成

窗口构造说明:
- `train_start` ~ `train_end`: 训练期 (约 488 个交易日 = 2 年)
- `train_end` ~ `test_start`: gap 15 个交易日 (避免 FWD_RET_10D 标签泄露)
- `test_start` ~ `test_end`: 测试期 (约 6 个月)

In [4]:
all_dates = np.sort(df['trade_date'].unique())
date_to_idx = {d: i for i, d in enumerate(all_dates)}
first_test = pd.Timestamp('2018-01-03')

windows = []
i = 0
while True:
    tsa = first_test + pd.DateOffset(months=6*i)
    tea = first_test + pd.DateOffset(months=6*(i+1))
    cs = all_dates[all_dates >= np.datetime64(tsa)]
    ce = all_dates[all_dates >= np.datetime64(tea)]
    if len(cs) == 0: break
    ts = cs[0]
    te = ce[0] if len(ce) > 0 else all_dates[-1]
    tsi = date_to_idx[ts]
    if tsi - 15 < 0: i += 1; continue
    tei = tsi - 15
    tsi2 = tei - 488
    if tsi2 < 0: i += 1; continue
    windows.append({'train_start': all_dates[tsi2], 'train_end': all_dates[tei],
                    'test_start': ts, 'test_end': te, 'window_id': i})
    i += 1

print(f'Windows: {len(windows)}')
for w in windows:
    print(f'  W{w["window_id"]:02d}: Train {pd.Timestamp(w["train_start"]).strftime("%Y-%m-%d")} ~ '
          f'{pd.Timestamp(w["train_end"]).strftime("%Y-%m-%d")} | Test {pd.Timestamp(w["test_start"]).strftime("%Y-%m-%d")} ~ '
          f'{pd.Timestamp(w["test_end"]).strftime("%Y-%m-%d")}')

Windows: 17
  W00: Train 2015-12-14 ~ 2017-12-12 | Test 2018-01-03 ~ 2018-07-03
  W01: Train 2016-06-08 ~ 2018-06-11 | Test 2018-07-03 ~ 2019-01-03
  W02: Train 2016-12-12 ~ 2018-12-11 | Test 2019-01-03 ~ 2019-07-03
  W03: Train 2017-06-12 ~ 2019-06-12 | Test 2019-07-03 ~ 2020-01-03
  W04: Train 2017-12-11 ~ 2019-12-12 | Test 2020-01-03 ~ 2020-07-03
  W05: Train 2018-06-07 ~ 2020-06-10 | Test 2020-07-03 ~ 2021-01-04
  W06: Train 2018-12-07 ~ 2020-12-11 | Test 2021-01-04 ~ 2021-07-05
  W07: Train 2019-06-11 ~ 2021-06-11 | Test 2021-07-05 ~ 2022-01-04
  W08: Train 2019-12-09 ~ 2021-12-13 | Test 2022-01-04 ~ 2022-07-04
  W09: Train 2020-06-05 ~ 2022-06-13 | Test 2022-07-04 ~ 2023-01-03
  W10: Train 2020-12-08 ~ 2022-12-12 | Test 2023-01-03 ~ 2023-07-03
  W11: Train 2021-06-04 ~ 2023-06-08 | Test 2023-07-03 ~ 2024-01-03
  W12: Train 2021-12-08 ~ 2023-12-12 | Test 2024-01-03 ~ 2024-07-03
  W13: Train 2022-06-08 ~ 2024-06-12 | Test 2024-07-03 ~ 2025-01-03
  W14: Train 2022-12-07 ~ 2024-12-12

## 3. 工具函数

In [5]:
def get_date_group_offsets(dates):
    """按日期分组, 返回 (starts, ends, counts)."""
    _, counts = np.unique(dates, return_counts=True)
    starts = np.cumsum(np.concatenate([[0], counts[:-1]]))
    return starts, starts + counts, counts


def compute_ics(preds, y, ts, te, min_size=10):
    """计算截面 Rank IC 列表 (argsort 快速 Spearman)."""
    ics = []
    for k in range(len(ts)):
        s, e = int(ts[k]), int(te[k])
        if e - s <= min_size:
            continue
        ic = np.corrcoef(
            np.argsort(np.argsort(preds[s:e])).astype(np.float64),
            np.argsort(np.argsort(y[s:e])).astype(np.float64)
        )[0, 1]
        if not np.isnan(ic):
            ics.append(ic)
    return ics


def ic_aware_obj(starts, ends, min_size=5):
    """IC-aware 自定义目标函数: 最小化负 Pearson 相关.
    
    梯度: grad = -(y_c - r * p_c) / (||y_c|| * ||p_c||)
    注意: starts/ends 必须来自训练集, 不能用测试集.
    """
    n = len(starts)
    def obj(preds, dtrain):
        y = dtrain.get_label().astype(np.float32)
        p = preds.astype(np.float32)
        g = np.zeros(len(p), np.float32)
        h = np.ones(len(p), np.float32)
        for k in range(n):
            s, e = int(starts[k]), int(ends[k])
            if e - s < min_size:
                continue
            yc = y[s:e] - y[s:e].mean()
            pc = p[s:e] - p[s:e].mean()
            ny = np.sqrt((yc * yc).sum()) + 1e-8
            np_ = np.sqrt((pc * pc).sum()) + 1e-8
            r = (yc * pc).sum() / (ny * np_)
            inv = 1.0 / (ny * np_)
            g[s:e] = -(yc - r * pc) * inv
            h[s:e] = inv
        return g, h
    return obj


def neg_ic_eval(starts, ends, min_size=10):
    """Early stopping 评估: 负 IC 均值.
    
    注意: starts/ends 必须来自验证集 (训练期尾部), 不能用测试集.
    """
    n = len(starts)
    def ev(preds, dtrain):
        y = dtrain.get_label()
        p = preds.astype(np.float64)
        ics, v = np.empty(n, np.float64), 0
        for k in range(n):
            s, e = int(starts[k]), int(ends[k])
            if e - s <= min_size:
                continue
            ic = np.corrcoef(
                np.argsort(np.argsort(p[s:e])).astype(np.float64),
                np.argsort(np.argsort(y[s:e])).astype(np.float64)
            )[0, 1]
            if not np.isnan(ic):
                ics[v] = ic
                v += 1
        return 'neg_ic', float(-ics[:v].mean()) if v > 0 else 0.0
    return ev

print('工具函数就绪')

工具函数就绪


## 4. 预计算窗口数据

对每个窗口，将训练集按时间 80/20 分割为子训练集和验证集。
- **子训练集**: 训练集前 80%，用于拟合模型
- **验证集**: 训练集后 20%，仅用于 early stopping 选树数量
- **测试集**: 完全不参与训练和调参

In [6]:
%%time
dates_all = df['trade_date'].values
all_feat = df[feature_cols].values.astype(np.float32)
all_tgt = df[target_col].values.astype(np.float32)

dp = {}
for i, d in enumerate(dates_all):
    dp.setdefault(d, []).append(i)

VAL_RATIO = 0.2  # 训练集后 20% 做验证

WD = {}
for w in windows:
    tr = np.array(sorted(p for d, ps in dp.items()
                         if w['train_start'] <= d <= w['train_end'] for p in ps), np.int64)
    te = np.array(sorted(p for d, ps in dp.items()
                         if w['test_start'] <= d <= w['test_end'] for p in ps), np.int64)
    vm_tr, vm_te = ~np.isnan(all_tgt[tr]), ~np.isnan(all_tgt[te])

    # 训练集内部 80/20 分割 (按时间)
    n_train = int(vm_tr.sum())
    split = int(n_train * (1 - VAL_RATIO))
    vm_sub = vm_tr.copy()
    vm_sub[split:] = False   # 子训练 mask
    vm_val = vm_tr.copy()
    vm_val[:split] = False   # 验证 mask

    # 子训练集
    X_sub = np.ascontiguousarray(all_feat[tr][vm_sub])
    y_sub = all_tgt[tr][vm_sub]
    d_sub = dates_all[tr][vm_sub]
    sub_s, sub_e, _ = get_date_group_offsets(d_sub)

    # 验证集 (来自训练期尾部)
    X_val = np.ascontiguousarray(all_feat[tr][vm_val])
    y_val = all_tgt[tr][vm_val]
    d_val = dates_all[tr][vm_val]
    val_s, val_e, _ = get_date_group_offsets(d_val)

    # 测试集
    X_te = np.ascontiguousarray(all_feat[te][vm_te])
    y_te = all_tgt[te][vm_te]
    d_te = dates_all[te][vm_te]
    te_s, te_e, _ = get_date_group_offsets(d_te)

    WD[w['window_id']] = {
        # 子训练
        'X_sub': X_sub, 'y_sub': y_sub, 'd_sub': d_sub,
        'sub_s': sub_s, 'sub_e': sub_e,
        # 验证 (训练期尾部)
        'X_val': X_val, 'y_val': y_val, 'd_val': d_val,
        'val_s': val_s, 'val_e': val_e,
        # 测试
        'X_te': X_te, 'y_te': y_te, 'd_te': d_te,
        'te_s': te_s, 'te_e': te_e,
    }

del df, all_feat, all_tgt, dates_all, dp
gc.collect()
print(f'Precomputed {len(windows)} windows')
print(f'防泄露: 子训练集拟合 + 验证集仅 early stopping, 无全量重训')
w0 = WD[0]
print(f'  示例 W00: sub_train={w0["X_sub"].shape[0]:,}  '
      f'val={w0["X_val"].shape[0]:,}  test={w0["X_te"].shape[0]:,}')

Precomputed 17 windows
防泄露: 子训练集拟合 + 验证集仅 early stopping, 无全量重训
  示例 W00: sub_train=111,530  val=27,896  test=34,381
CPU times: total: 4.02 s
Wall time: 4.17 s


## 5. 模型训练

### 训练流程 (每窗口) — 最保守版

```
Step 1: 在子训练集 (80%) 上训练, 用验证集 (20%) early stopping → 最终模型
Step 2: 直接用该模型在测试集上预测, 计算 IC/ICIR
```

### 防泄露要点
- 验证集仅用于 early stopping 指标评估, 不参与梯度计算
- 不做全量重训, 避免验证集数据流入模型权重
- 测试集完全隔离, 从不参与任何训练环节

### 最优超参
| 参数 | 值 | 含义 |
|------|-----|------|
| learning_rate | 0.1684 | 学习率 |
| max_depth | 3 | 树深度 |
| min_child_weight | 129.07 | 叶节点最小权重 |
| colsample_bytree | 0.55 | 特征采样比例 |
| subsample | 0.62 | 样本采样比例 |
| reg_alpha | 3.27 | L1 正则化 |
| reg_lambda | 2.99 | L2 正则化 |

In [7]:
# 最优参数 (07_ablation Phase 2 精调)
GBM_PARAMS = {
    'max_depth': 3,
    'eta': 0.1683783001073622,
    'min_child_weight': 129.06560487175187,
    'colsample_bytree': 0.5496012159790593,
    'subsample': 0.6169232204173679,
    'reg_alpha': 3.267822756002464,
    'reg_lambda': 2.9945154774416065,
    'seed': RANDOM_STATE,
    'tree_method': 'hist',
    'device': 'cuda',
    'nthread': N_JOBS,
}

MIN_TREES = 50  # B1: 强制最少 50 棵树
MAX_TREES = 1000
EARLY_STOP = 30

print('GBM params ready')

GBM params ready


In [8]:
%%time
print('=' * 75)
print('XGBoost GBM (B1 MinTrees) — 保守版: 无全量重训')
print('=' * 75)
print(f'{"W":>3}  {"Test Period":>24}  {"IC":>8}  {"ICIR":>8}  {"Trees":>6}')
print('-' * 60)

results = []
t_total = time.time()

for w in windows:
    wid = w['window_id']
    wd = WD[wid]
    t0 = time.time()

    # ── Step 1: 子训练 + 验证集 early stopping → 找最优树数 ──
    d_sub = xgb.QuantileDMatrix(wd['X_sub'], label=wd['y_sub'])
    d_val = xgb.QuantileDMatrix(wd['X_val'], label=wd['y_val'])

    obj_sub = ic_aware_obj(wd['sub_s'], wd['sub_e'])
    evl_val = neg_ic_eval(wd['val_s'], wd['val_e'])

    # 强制热身 MIN_TREES 棵
    m_search = xgb.train(GBM_PARAMS, d_sub, MIN_TREES, obj=obj_sub, verbose_eval=False)
    # 继续 early stopping
    m_search = xgb.train(GBM_PARAMS, d_sub, MAX_TREES - MIN_TREES,
                         obj=obj_sub, evals=[(d_val, 'v')], custom_metric=evl_val,
                         early_stopping_rounds=EARLY_STOP, verbose_eval=False,
                         xgb_model=m_search)
    best_n_trees = m_search.best_iteration + MIN_TREES + 1
    del d_sub, d_val

    # ── Step 2: 直接用 early stopping 模型预测 (不做全量重训) ──
    d_test = xgb.DMatrix(wd['X_te'])
    preds = m_search.predict(d_test)
    del d_test, m_search

    ics = compute_ics(preds, wd['y_te'], wd['te_s'], wd['te_e'])
    mic = np.mean(ics) if ics else 0.0
    sic = np.std(ics) if ics else 0.0
    icir = mic / sic if sic > 1e-8 else 0.0

    results.append({
        'wid': wid, 'mic': mic, 'icir': icir, 'n_trees': best_n_trees,
    })

    ts_str = pd.Timestamp(w['test_start']).strftime('%Y-%m-%d')
    te_str = pd.Timestamp(w['test_end']).strftime('%Y-%m-%d')
    dt = time.time() - t0
    print(f'{wid:>3}  {ts_str} ~ {te_str}  {mic:>+8.4f}  {icir:>+8.4f}  {best_n_trees:>6}')

print(f'\nTotal: {time.time() - t_total:.1f}s')

XGBoost GBM (B1 MinTrees) — 保守版: 无全量重训
  W               Test Period        IC      ICIR   Trees
------------------------------------------------------------
  0  2018-01-03 ~ 2018-07-03   +0.0502   +0.5268     268
  1  2018-07-03 ~ 2019-01-03   +0.0439   +0.3764     238
  2  2019-01-03 ~ 2019-07-03   +0.0788   +0.4894     992
  3  2019-07-03 ~ 2020-01-03   +0.0118   +0.0863     158
  4  2020-01-03 ~ 2020-07-03   +0.0601   +0.4133     121
  5  2020-07-03 ~ 2021-01-04   +0.0619   +0.3986     118
  6  2021-01-04 ~ 2021-07-05   +0.0126   +0.0532     402
  7  2021-07-05 ~ 2022-01-04   -0.1007   -0.4575     101
  8  2022-01-04 ~ 2022-07-04   +0.0555   +0.5136     820
  9  2022-07-04 ~ 2023-01-03   +0.0471   +0.4492     116
 10  2023-01-03 ~ 2023-07-03   +0.0535   +0.3794     239
 11  2023-07-03 ~ 2024-01-03   +0.0801   +0.5978     108
 12  2024-01-03 ~ 2024-07-03   +0.0966   +0.6448     111
 13  2024-07-03 ~ 2025-01-03   +0.0720   +0.3494     104
 14  2025-01-03 ~ 2025-07-03   +0.0873   +0.

## 6. 结果汇总

In [9]:
# 逐折结果
print('=' * 75)
print('逐折结果')
print('=' * 75)
print(f'{"W":>3}  {"Test Period":>24}  {"IC":>8}  {"ICIR":>8}  {"Trees":>6}')
print('-' * 60)
for r in results:
    w = windows[r['wid']]
    ts_str = pd.Timestamp(w['test_start']).strftime('%Y-%m-%d')
    te_str = pd.Timestamp(w['test_end']).strftime('%Y-%m-%d')
    print(f'{r["wid"]:>3}  {ts_str} ~ {te_str}  {r["mic"]:>+8.4f}  {r["icir"]:>+8.4f}  {r["n_trees"]:>6}')

# 总汇总: 各窗口指标的简单平均
mics = [r['mic'] for r in results]
icirs = [r['icir'] for r in results]
trees = [r['n_trees'] for r in results]
avg_ic = np.mean(mics)
avg_icir = np.mean(icirs)
ic_pos = sum(1 for x in mics if x > 0)
n = len(mics)
avg_trees = np.mean(trees)

print('-' * 60)
print(f'{"Avg":>3}  {"":>24}  {avg_ic:>+8.4f}  {avg_icir:>+8.4f}  {avg_trees:>6.0f}')
print()
print(f'平均 IC:   {avg_ic:+.4f}')
print(f'平均 ICIR: {avg_icir:+.4f}')
print(f'IC>0:      {ic_pos}/{n} ({ic_pos/n*100:.1f}%)')
print(f'平均树数:  {avg_trees:.0f}')

逐折结果
  W               Test Period        IC      ICIR   Trees
------------------------------------------------------------
  0  2018-01-03 ~ 2018-07-03   +0.0502   +0.5268     268
  1  2018-07-03 ~ 2019-01-03   +0.0439   +0.3764     238
  2  2019-01-03 ~ 2019-07-03   +0.0788   +0.4894     992
  3  2019-07-03 ~ 2020-01-03   +0.0118   +0.0863     158
  4  2020-01-03 ~ 2020-07-03   +0.0601   +0.4133     121
  5  2020-07-03 ~ 2021-01-04   +0.0619   +0.3986     118
  6  2021-01-04 ~ 2021-07-05   +0.0126   +0.0532     402
  7  2021-07-05 ~ 2022-01-04   -0.1007   -0.4575     101
  8  2022-01-04 ~ 2022-07-04   +0.0555   +0.5136     820
  9  2022-07-04 ~ 2023-01-03   +0.0471   +0.4492     116
 10  2023-01-03 ~ 2023-07-03   +0.0535   +0.3794     239
 11  2023-07-03 ~ 2024-01-03   +0.0801   +0.5978     108
 12  2024-01-03 ~ 2024-07-03   +0.0966   +0.6448     111
 13  2024-07-03 ~ 2025-01-03   +0.0720   +0.3494     104
 14  2025-01-03 ~ 2025-07-03   +0.0873   +0.5049     104
 15  2025-07-03 ~ 202